# Chapter 12：面向 LLM 的强化学习与 Open R1

欢迎来到 Chapter 12！本章将带你深入了解强化学习（Reinforcement Learning）在大型语言模型中的应用，以及 Open R1 这一开源项目。

> **TIP**: Open R1 是 HuggingFace 发起的社区项目，旨在让先进的 AI 推理能力对所有人开放。本章的目标是帮助学习者理解、使用并贡献 Open R1。

## 本章概览

### 背景：LLM 的推理局限

大型语言模型（LLM）在许多生成任务上表现出色，但在需要**多步推理**的复杂问题上长期表现不佳。例如：
- 数学证明题
- 逻辑谜题
- 多步骤数学计算

Open R1 通过**强化学习**训练模型先进行「思考」再给出答案，让 LLM 能够处理这类复杂推理问题。

### 推理模型的输出格式

经过强化学习训练后，模型会生成结构化的「思考 + 答案」格式：

```
问题：我有 3 个苹果和 2 个橙子，总共有多少个水果？

<think>
我需要把苹果数量和橙子数量加起来，得到水果总数。
</think>

<answer>
5
</answer>
```

用户可以分别提取 `<think>` 和 `<answer>` 部分，分别处理推理过程和最终答案。

### 本章学习路线

| 节次 | 主题 |
|------|------|
| Section 01 | 强化学习简介 + RL 在 LLM 中的应用 |
| Section 02 | DeepSeek R1 论文解析 |
| Section 03 | GRPO 数学原理深度解析 |
| Section 04 | 在 TRL 中实现 GRPO |
| Section 05 | 实战练习：GRPO 微调（SmolLM2） |
| Section 06 | 实战练习：Unsloth 加速 GRPO |

### 预备知识

建议在学习本章前掌握：
- Python 编程基础
- 机器学习基本概念
- 已完成 Chapter 1–11 的内容

## 强化学习（Reinforcement Learning）简介

### 用一个类比理解 RL

想象你在训练一只狗学会「坐下」：
- 你说「坐下！」
- 如果狗坐下了，给它零食和表扬（**正向奖励**）
- 如果它没坐下，重新引导它
- 反复练习，狗逐渐学会将「坐下」动作与正向奖励关联

这就是强化学习的核心思想：**通过奖励信号引导 Agent 学习最优行为策略**。

在 LLM 的语境中：
- 「狗」= 语言模型（**Agent**）
- 「主人」= 提供反馈的环境（**Environment**）
- 「零食」= 奖励信号（**Reward**）

### 强化学习的核心概念

#### Agent（智能体）
负责学习和决策的主体。在 LLM 场景中，**语言模型本身就是 Agent**。

#### Environment（环境）
Agent 所处并与之交互的世界。对于 LLM，环境可能是用户交互场景、数学验证程序等。

#### Action（动作）
Agent 在环境中可以做出的选择。对于 LLM，动作是**生成 token**，也就是生成文字回复。

#### Reward（奖励）
环境在 Agent 执行动作后给出的反馈，通常是一个数值：
- **正向奖励**：告诉 Agent「做得好！」
- **负向奖励（惩罚）**：告诉 Agent「这样不对，换个方式」

#### Policy（策略）
Agent 选择动作的策略/规则。RL 训练的本质就是**优化 Policy**，让 Agent 在各种情况下都能选择获得更高奖励的动作。

### RL 训练过程：试错循环

| 步骤 | 名称 | 描述 |
|------|------|------|
| 1 | 观察（Observation） | Agent 感知当前环境状态 |
| 2 | 行动（Action） | Agent 根据当前 Policy 决定采取什么动作 |
| 3 | 反馈（Feedback） | 环境给出奖励信号 |
| 4 | 学习（Learning） | Agent 更新 Policy：强化带来高奖励的动作 |
| 5 | 迭代（Iteration） | 重复以上过程直到收敛 |

> **TIP**: 这就像学骑自行车：摔倒是负奖励，顺利骑行是正奖励。你通过反复尝试调整平衡和踩踏力度，最终掌握技能。

In [1]:
# 强化学习核心概念演示：用 Python 模拟简单的奖励函数

def simple_reward(response: str, correct_answer: str) -> float:
    """
    简单的二元奖励函数：
    - 回答正确：奖励 1.0
    - 回答错误：奖励 0.0
    
    在真实的 GRPO 训练中，奖励函数可以更复杂，
    例如考虑格式、推理步骤、答案准确度等多个维度
    """
    # 去除首尾空白后比较答案
    if response.strip() == correct_answer.strip():
        return 1.0  # 正确答案，给予正向奖励
    else:
        return 0.0  # 错误答案，无奖励


# 模拟模型对一道数学题给出的多个候选回答
question = "2 + 2 × 6 = ?"
correct_answer = "14"  # 按照运算优先级：先乘法 2×6=12，再加法 2+12=14

# 模型生成的 4 个候选答案（模拟 GRPO 的 group sampling）
model_responses = ["14", "16", "10", "14"]

print(f"问题：{question}")
print(f"正确答案：{correct_answer}")
print()
print("各候选答案的奖励值：")

rewards = []
for i, response in enumerate(model_responses):
    reward = simple_reward(response, correct_answer)
    rewards.append(reward)
    status = "✓ 正确" if reward == 1.0 else "✗ 错误"
    print(f"  候选 {i+1}: '{response}' → 奖励 = {reward} ({status})")

print()
print(f"平均奖励：{sum(rewards) / len(rewards):.2f}")
print(f"正确率：{sum(rewards) / len(rewards) * 100:.0f}%")

问题：2 + 2 × 6 = ?
正确答案：14

各候选答案的奖励值：
  候选 1: '14' → 奖励 = 1.0 (✓ 正确)
  候选 2: '16' → 奖励 = 0.0 (✗ 错误)
  候选 3: '10' → 奖励 = 0.0 (✗ 错误)
  候选 4: '14' → 奖励 = 1.0 (✓ 正确)

平均奖励：0.50
正确率：50%


## RL 在大型语言模型中的作用

### 预训练的局限性

通过海量文本预训练的 LLM 擅长预测下一个 token，能生成流畅的文字。但「流畅」不等于「好用」。我们希望 LLM 还能做到：

- **有帮助（Helpful）**：提供有用的、相关的信息
- **无害（Harmless）**：避免生成有毒、偏见或有害的内容
- **符合人类偏好（Aligned）**：以人类觉得自然、有益的方式回应

纯粹的预训练和监督微调（SFT）在这些方面往往不够：微调后的模型可能生成流畅但**事实错误**、或**回避问题**的回答。

### 基于人类反馈的强化学习（RLHF）

RLHF（Reinforcement Learning from Human Feedback）是当前最流行的 LLM 对齐技术：

**Step 1：收集人类偏好数据**
让人类比较同一问题的多个回答，选出更好的那个。

**Step 2：训练奖励模型（Reward Model）**
用人类偏好数据训练一个独立的奖励模型，它能预测哪类回答更符合人类偏好。

**Step 3：用 RL 微调 LLM**
以奖励模型作为「环境」，用强化学习微调 LLM，让它生成奖励模型认为好的回答。

| RLHF 的好处 | 说明 |
|-------------|------|
| 更强的可控性 | 可以引导模型生成符合特定目标的文本（更有帮助、更简洁等） |
| 与人类价值观对齐 | 通过人类判断来对齐主观偏好，难以用规则描述 |
| 减少有害行为 | 通过负向奖励惩罚有毒内容、偏见、错误信息等 |

> **TIP**: GPT-4、Gemini、DeepSeek R1 等主流 LLM 都使用了 RLHF 或其变体。

## GRPO：本章的核心算法

RLHF 有许多具体实现算法，本章重点介绍 **GRPO（Group Relative Policy Optimization，组相对策略优化）**。

### 常见 RLHF 算法对比

| 算法 | 特点 | 主要局限 |
|------|------|----------|
| **PPO**（Proximal Policy Optimization） | 最早的高效 RLHF 方法，使用策略梯度更新 | 需要独立的价值网络（Critic），计算成本高 |
| **DPO**（Direct Preference Optimization） | 不需要奖励模型，直接用偏好数据训练 | 需要成对偏好数据，泛化能力有限 |
| **GRPO**（Group Relative Policy Optimization） | 对一组生成结果进行组内比较，无需 Critic 网络 | 需要生成多个候选响应，计算量略高 |

### 为什么选择 GRPO？

GRPO 的核心创新在于：

1. **组内比较（Group-based Learning）**：为同一个 prompt 生成多个候选回答（通常 4~16 个），通过组内相对比较来确定哪些回答更好，比逐对比较（pairwise）更稳定

2. **无需独立 Critic 网络**：传统 PPO 需要训练一个与 policy 同等大小的价值网络，GRPO 通过组内奖励归一化替代了这一功能，大幅节省计算资源

3. **奖励函数灵活**：不局限于奖励模型，可以使用任何可返回数值的函数：
   - 数学验证函数（验证答案是否正确）
   - 格式检查函数（是否符合 XML 格式）
   - 长度控制函数（是否达到目标长度）

> **TIP**: GRPO 是 DeepSeek R1 论文中提出的关键算法，DeepSeek 团队在此基础上训练出了具备强大推理能力的模型。

In [2]:
# 演示 GRPO 的核心思想：组内相对优势（Advantage）计算
import statistics

def compute_group_advantages(rewards: list) -> list:
    """
    GRPO 的优势计算：对同一组内的奖励进行归一化
    
    公式：advantage_i = (reward_i - mean(rewards)) / std(rewards)
    
    归一化后：
    - advantage > 0：该回答优于组内平均水平 → 应被强化
    - advantage < 0：该回答低于组内平均水平 → 应被抑制
    """
    mean_reward = statistics.mean(rewards)   # 组内平均奖励
    std_reward = statistics.stdev(rewards)   # 组内奖励标准差
    
    # 计算每个回答的相对优势
    advantages = [(r - mean_reward) / (std_reward + 1e-8) for r in rewards]
    return advantages


# 模拟：对同一道数学题，模型生成 8 个候选答案
# 奖励：正确答案得 1，错误答案得 0
rewards_group1 = [1, 0, 0, 1, 1, 0, 1, 0]  # 4 个正确，4 个错误

advantages = compute_group_advantages(rewards_group1)

print("=== GRPO 组内优势计算示例 ===")
print(f"原始奖励：{rewards_group1}")
print(f"组内均值：{statistics.mean(rewards_group1):.4f}")
print(f"组内标准差：{statistics.stdev(rewards_group1):.4f}")
print()
print("各候选答案的优势值：")
for i, (reward, adv) in enumerate(zip(rewards_group1, advantages)):
    direction = "↑ 强化" if adv > 0 else "↓ 抑制"
    print(f"  候选 {i+1}: 奖励={reward}, 优势={adv:+.4f} → {direction}")

print()
print("核心洞察：")
print("  GRPO 不关注绝对奖励值，而是通过组内比较")
print("  来判断哪些回答相对更好，从而指导模型学习方向")

=== GRPO 组内优势计算示例 ===
原始奖励：[1, 0, 0, 1, 1, 0, 1, 0]
组内均值：0.5000
组内标准差：0.5345

各候选答案的优势值：
  候选 1: 奖励=1, 优势=+0.9354 → ↑ 强化
  候选 2: 奖励=0, 优势=-0.9354 → ↓ 抑制
  候选 3: 奖励=0, 优势=-0.9354 → ↓ 抑制
  候选 4: 奖励=1, 优势=+0.9354 → ↑ 强化
  候选 5: 奖励=1, 优势=+0.9354 → ↑ 强化
  候选 6: 奖励=0, 优势=-0.9354 → ↓ 抑制
  候选 7: 奖励=1, 优势=+0.9354 → ↑ 强化
  候选 8: 奖励=0, 优势=-0.9354 → ↓ 抑制

核心洞察：
  GRPO 不关注绝对奖励值，而是通过组内比较
  来判断哪些回答相对更好，从而指导模型学习方向


## RL 学习方式的演进

下图展示了从预训练到强化学习对齐的整体流程：

```
预训练（Pre-training）
    ↓ 大规模文本数据，学习语言知识
    
监督微调（SFT - Supervised Fine-Tuning）
    ↓ 高质量指令数据，学习遵循指令  [见 Chapter 11]
    
强化学习对齐（RL Alignment）
    ↓ 奖励信号引导，学习人类偏好  [本章重点]
    
推理增强（Reasoning Enhancement）
    ↓ GRPO + 可验证任务，发展推理能力  [本章核心]
```

> **TIP**: 监督微调擅长产生结构化输出，但在「有帮助、无害、符合人类偏好」这些维度上效果有限。RL 正是为了补充 SFT 的不足而引入的。

## 本节小结

### 核心要点

1. **强化学习**通过奖励信号引导 Agent 学习最优策略，核心组件包括：Agent、Environment、Action、Reward、Policy

2. **RLHF** 使用人类偏好数据作为奖励信号，训练 LLM 生成更符合人类期望的回答

3. **GRPO** 是本章的核心算法，其创新点在于：
   - 组内相对比较（无需 Critic 网络）
   - 支持任意奖励函数
   - 训练更稳定、计算更高效

4. **Open R1** 是基于 GRPO 的开源项目，目标是让 LLM 具备强大的推理能力

### 思考练习

1. 强化学习的核心组成要素有哪些？
   Agent, Environment, State, Action, Reward, Policy（+ Value）
   state → action → reward → next state
2. RLHF 对训练语言模型的主要优势是什么？
   更好的可控性、人类价值观对齐、减少有害行为
3. 在 RL for LLMs 的语境中，什么代表「动作（Action）」？
   生成一个 token（逐步决策）
4. GRPO 相比 PPO 的关键区别是什么？
   GRPO 用 group 内相对优势替代 value function，从而避免训练 critic
5. 为什么可以用数学验证函数代替奖励模型来训练 GRPO？
   数学验证函数提供一个 可计算的真实奖励（oracle reward），从而不需要训练 reward model


---

**下一节**：我们将深入解析 DeepSeek R1 论文，了解 GRPO 如何在实际中训练出具备强推理能力的模型。

---

## 问答记录

本节汇总学习过程中产生的有价值问题及解答，供复习参考。

---

### Q1：GRPO 例子中奖励非 0 即 1，组内优势差异是否太小？

**问题背景**：在 GRPO 的演示代码中，原始奖励为二元值（正确=1，错误=0）。这种情况下，组内所有正确答案的 advantage 相同、所有错误答案的 advantage 也相同，差异似乎不够细粒度。这是例子不恰当，还是有其他机制补偿？

**答：两者都有——例子略为简化，但二元奖励本身也有其合理性和局限性。**

#### 局限性确实存在

二元奖励下，组内只产生两种 advantage 值（约 ±0.935），无法区分：
- 「推理过程清晰但答案凑巧正确」vs「直接猜对」
- 「格式规范的错误回答」vs「逻辑混乱的错误回答」

梯度只告诉模型「对/错」方向，无法提供细粒度的质量信号。

#### 真实 GRPO 用多维复合奖励解决

DeepSeek R1 实际使用的是**多维连续奖励**，而非单一二元值：

```python
def real_reward(response, correct_answer):
    score = 0.0

    # ① 格式奖励（连续，0~0.5）：鼓励模型输出结构化的推理过程
    if has_think_tag(response):         score += 0.25
    if has_answer_tag(response):        score += 0.25

    # ② 准确性奖励（二元，0 or 1）
    if extract_answer(response) == correct_answer:
        score += 1.0

    # ③ 可选：推理步骤质量、长度惩罚等
    return score
    # → 实际奖励范围约 [0, 1.5]，组内差异远大于纯二元情况
```

这样同组内会出现如 `[1.5, 0.25, 1.0, 0.5, 0.0, 1.25]` 的分布，advantage 有细粒度差异，梯度信号更丰富。

#### 二元奖励的一个隐藏的「正确设计」

当一组 G 个回答**全对或全错**时，标准差 = 0，所有 advantage ≈ 0，**梯度更新自动归零**：

```python
rewards_all_correct = [1, 1, 1, 1, 1, 1, 1, 1]
# std = 0 → advantage ≈ 0 → 不浪费梯度步（模型已掌握，无需更新）

rewards_all_wrong = [0, 0, 0, 0, 0, 0, 0, 0]
# std = 0 → advantage ≈ 0 → 不乱更新（无正例可参考，更新无意义）
```

这是 GRPO 的优雅之处：**只在组内「有分歧」的 prompt 上产生学习信号，自动跳过已掌握或完全不会的样本**。

#### 结论

| 维度 | 说明 |
|------|------|
| 示例代码的局限 | 只展示了信号存在，未体现粒度差异 |
| 真实训练的做法 | 多维复合奖励（格式 + 准确性）提供连续信号 |
| 二元奖励仍有效 | 梯度方向（正/负）比幅度更关键；全同组自动静默 |
| 核心机制不变 | 组内相对比较替代绝对打分，这一设计本身是正确的 |